In [0]:
%pip install dice-ml fairlearn

In [0]:
import numpy as np
import pandas as pd

from fairlearn.metrics import MetricFrame, demographic_parity_difference, selection_rate
from fairlearn.datasets import fetch_adult
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from fairlearn.reductions import GridSearch, DemographicParity

In [0]:
data_dictionary = fetch_adult(as_frame= True)


In [0]:
X.head()

In [0]:
SENSITIVE_VAR = 'sex'
X = data_dictionary.data
y = (data_dictionary.target == ">50K").astype(int)
S = X[SENSITIVE_VAR]

X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(X, y, S, test_size = 0.2, random_state= 0)

In [0]:
y

In [0]:
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']

preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', StandardScaler(), numerical_features),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [0]:
lr = LogisticRegression()
lr.fit(X_train_processed, y_train)

y_test_unmitigated = lr.predict(X_test_processed)

In [0]:
metrics_fns = {
    'accuracy': accuracy_score,
    'selection_rate': selection_rate,
    
}
unmitigated_metrics = MetricFrame(metrics = metrics_fns,
                          y_true= y_test,
                          y_pred= y_test_unmitigated,
                          sensitive_features= S_test)
unmitigated_metrics.overall

In [0]:
## Interpretación
## Tenemos un promedio de predicciones positivas de 19% (la categoría dominante es 'hombre' en SENSITIVE_FEATURES)

In [0]:
demographic_parity_difference(y_test, y_test_unmitigated, sensitive_features= S_test)

In [0]:
## Interpretación:
## Hay una diferencia de 17.6 puntos porcentuales entre grupos
## Grupo	Selection Rate
## Hombres	    25%
## Mujeres	    7.4%
## 👉 Diferencia: ~17.6%

🚨 Insight de experto (CRÍTICO)

✔️ Regla práctica:
- < 0.05 → OK
- 0.05 – 0.10 → warning
- 0.10 → ⚠️ problema serio de fairness

👉 Tu caso:

> 0.176 → ALTO SESGO

⚠️ Qué significa esto en la práctica

Tu modelo:

- Está tomando decisiones diferentes según sex
- Podría estar discriminando indirectamente

Esto puede pasar por:

🔻 Causas típicas
- Features correlacionadas con sexo
- ingresos
- tipo de trabajo
- historial crediticio
- Data imbalance histórico
- mujeres con menos crédito aprobado históricamente
- Threshold único
- un solo threshold afecta distinto a cada grupo

In [0]:
## Si el grupo de selección (hombres) por ejemplo es de 19% y otro que seleccionemos que es de 1%, la diferencía es muy grande. Tendría fairness, ya que la selección de un grupo particular pues está teniendo representitividad muy baja.
# Paso 1: Seleccionamos un grupo de métricas. El selection_rate lo que hace es medir predicciones positivas (que son 1 = >50k, prob. de una persona gane más de 50k). Lo que nos interesaría medir es el que el modelo NO tenga un sesgo en función de una variable que nosotros consideremos sensible (nuestro caso fue la variable 'sex'). Esta variable sensible no se incluye dentro del entrenamiento del modelo. Nos interesa ver si hay algun tipo de problema de fearness cuando medimos sobre una variable particular el modelo, pero tenemos que tener una variable de referencia, esa variable de referencia es 'sex'.
# Paso 2: Le pasamos las variables sensibles, que puede ser más de una.

In [0]:
unmitigated_metrics.by_group


### Fair Model

In [0]:
constrains = DemographicParity()

gs = GridSearch(
  LogisticRegression(),
  constraints= constrains,
  constraint_weight= 0.5,
)

X_train_dense = X_train_processed.toarray()
gs.fit(X_train_dense, y_train, sensitive_features = S_train)

In [0]:
y_pred_metidated = gs.predict(X_test_processed.toarray())
mitigated_metrics = MetricFrame(metrics = metrics_fns,
                          y_true= y_test,
                          y_pred= y_pred_metidated,
                          sensitive_features= S_test)

In [0]:
mitigated_metrics.by_group

In [0]:
mitigated_metrics.overall